FAISS

In [4]:
# Facebook AI similarity search (faiss) is a library for efficient similarity search
# and clustering of dense vectors. It contains algorithms that search in sets of
# vectors of any size, up to ones that possibly do not fit in RAM. It also contains
# supporting code for evaluation and parameter tuning

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter


c:\Generative AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
loader = TextLoader("..\speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=15)
docs = text_splitter.split_documents(documents)


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Rajeev Pandey\AppData\Local\Temp\ipykernel_19780\4261439034.py:1: SyntaxWarning: invalid escape sequence '\s'
  loader = TextLoader("..\speech.txt")
Created a chunk of size 363, which is longer than the specified 200
Created a chunk of size 422, which is longer than the specified 200


In [7]:
docs

[Document(metadata={'source': '..\\speech.txt'}, page_content='Have you ever spent twenty minutes scrolling through a streaming service, overwhelmed by thousands of movies, only to give up and re-watch something youâ€™ve already seen? This isn\'t a personal failing; it\'s a phenomenon called the "paradox of choice." We assume that more choice equals more freedom, and therefore, more happiness. But the opposite is often true.'),
 Document(metadata={'source': '..\\speech.txt'}, page_content='When faced with an abundance of options, we don\'t feel liberated; we feel paralyzed. This happens for a few reasons. First, there\'s decision fatigue. Every choice we make, big or small, depletes our mental energy. Second, our expectations soar. We no longer want a good option; we feel we must find the perfect one. Finally, there\'s the constant fear of regretâ€”the nagging "what if" about all the choices we didn\'t make.'),
 Document(metadata={'source': '..\\speech.txt'}, page_content='This paradox

In [8]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

C:\Users\Rajeev Pandey\AppData\Local\Temp\ipykernel_19780\909259017.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [9]:
db = FAISS.from_documents(docs,embeddings)
db

In [10]:
# querying
query = "when faced with abundant options, what do we feel?"
doc = db.similarity_search(query)
doc[0].page_content


'When faced with an abundance of options, we don\'t feel liberated; we feel paralyzed. This happens for a few reasons. First, there\'s decision fatigue. Every choice we make, big or small, depletes our mental energy. Second, our expectations soar. We no longer want a good option; we feel we must find the perfect one. Finally, there\'s the constant fear of regretâ€”the nagging "what if" about all the choices we didn\'t make.'

Retriever

In [11]:
# Retrievers are like an interface  , which can be connected to vector store db when we convert the vectorstores into  a retriever class.
#  it can retrieve information from vectore store and provide response
#  it allows us to easily use it in other langchain methods which largely work with retrievers



In [12]:
retriever = db.as_retriever()
retriever.invoke(query)

[Document(id='1cbf601f-05f0-4f76-8b95-2fd5293251b2', metadata={'source': '..\\speech.txt'}, page_content='When faced with an abundance of options, we don\'t feel liberated; we feel paralyzed. This happens for a few reasons. First, there\'s decision fatigue. Every choice we make, big or small, depletes our mental energy. Second, our expectations soar. We no longer want a good option; we feel we must find the perfect one. Finally, there\'s the constant fear of regretâ€”the nagging "what if" about all the choices we didn\'t make.'),
 Document(id='cbc565f2-c9c7-4088-b72d-b2dc6dfcdab6', metadata={'source': '..\\speech.txt'}, page_content='This paradox extends far beyond our TV screens. It\'s in the endless scroll of social media, the countless products available online, and even the career paths we\'re told are possible. The secret to navigating this isn\'t to find the perfect choice, but to find freedom in making a choice. By embracing "good enough" and committing to a decision, we free ou

Similarity search with score

In [13]:
# there are some FAISS specific methods. One of them is similarity_search_with_score, which allows
#  you to return not only the documents but also the distance score of the query to them. The returened distance score is L2 distance.(manhattan distance)
# THerefore a lower score is better.

In [14]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='1cbf601f-05f0-4f76-8b95-2fd5293251b2', metadata={'source': '..\\speech.txt'}, page_content='When faced with an abundance of options, we don\'t feel liberated; we feel paralyzed. This happens for a few reasons. First, there\'s decision fatigue. Every choice we make, big or small, depletes our mental energy. Second, our expectations soar. We no longer want a good option; we feel we must find the perfect one. Finally, there\'s the constant fear of regretâ€”the nagging "what if" about all the choices we didn\'t make.'),
  np.float32(247.47662)),
 (Document(id='cbc565f2-c9c7-4088-b72d-b2dc6dfcdab6', metadata={'source': '..\\speech.txt'}, page_content='This paradox extends far beyond our TV screens. It\'s in the endless scroll of social media, the countless products available online, and even the career paths we\'re told are possible. The secret to navigating this isn\'t to find the perfect choice, but to find freedom in making a choice. By embracing "good enough" and committi

In [15]:
# passing vectors directly 
embedding_vector = embeddings.embed_query(query)
embedding_vector

[0.8193251490592957,
 1.738477110862732,
 -3.640289783477783,
 -0.6122704148292542,
 1.6644376516342163,
 0.6669067144393921,
 0.08611617237329483,
 0.30517396330833435,
 0.9252099394798279,
 0.646238386631012,
 0.3065188229084015,
 0.03133683279156685,
 1.3869268894195557,
 2.144669771194458,
 0.1605958789587021,
 0.12294431030750275,
 0.18322615325450897,
 -0.5461795330047607,
 0.5815341472625732,
 1.0921194553375244,
 -0.5312899947166443,
 -0.8416656255722046,
 -0.284504771232605,
 1.2673090696334839,
 0.043850481510162354,
 1.290644884109497,
 -0.3376520872116089,
 0.23671475052833557,
 0.21717455983161926,
 -0.3902689516544342,
 -0.11698602885007858,
 -0.36417779326438904,
 0.7755268216133118,
 0.06227432191371918,
 -0.9327678680419922,
 -0.4688223600387573,
 0.29072508215904236,
 0.412020206451416,
 0.6264081597328186,
 -1.1313855648040771,
 -0.299203485250473,
 -0.3344680964946747,
 1.0591098070144653,
 -1.6347987651824951,
 0.3768959641456604,
 -0.41573458909988403,
 0.81434512

In [17]:
docs_score=db.similarity_search_by_vector(embedding_vector)

In [18]:
docs_score

[Document(id='1cbf601f-05f0-4f76-8b95-2fd5293251b2', metadata={'source': '..\\speech.txt'}, page_content='When faced with an abundance of options, we don\'t feel liberated; we feel paralyzed. This happens for a few reasons. First, there\'s decision fatigue. Every choice we make, big or small, depletes our mental energy. Second, our expectations soar. We no longer want a good option; we feel we must find the perfect one. Finally, there\'s the constant fear of regretâ€”the nagging "what if" about all the choices we didn\'t make.'),
 Document(id='cbc565f2-c9c7-4088-b72d-b2dc6dfcdab6', metadata={'source': '..\\speech.txt'}, page_content='This paradox extends far beyond our TV screens. It\'s in the endless scroll of social media, the countless products available online, and even the career paths we\'re told are possible. The secret to navigating this isn\'t to find the perfect choice, but to find freedom in making a choice. By embracing "good enough" and committing to a decision, we free ou

In [19]:
# saving this db to my local disk
db.save_local("faiss_vectorstore")

In [22]:
new_df = FAISS.load_local("faiss_vectorstore", embeddings,allow_dangerous_deserialization=True)
docs = new_df.similarity_search(query)

In [23]:
docs

[Document(id='1cbf601f-05f0-4f76-8b95-2fd5293251b2', metadata={'source': '..\\speech.txt'}, page_content='When faced with an abundance of options, we don\'t feel liberated; we feel paralyzed. This happens for a few reasons. First, there\'s decision fatigue. Every choice we make, big or small, depletes our mental energy. Second, our expectations soar. We no longer want a good option; we feel we must find the perfect one. Finally, there\'s the constant fear of regretâ€”the nagging "what if" about all the choices we didn\'t make.'),
 Document(id='cbc565f2-c9c7-4088-b72d-b2dc6dfcdab6', metadata={'source': '..\\speech.txt'}, page_content='This paradox extends far beyond our TV screens. It\'s in the endless scroll of social media, the countless products available online, and even the career paths we\'re told are possible. The secret to navigating this isn\'t to find the perfect choice, but to find freedom in making a choice. By embracing "good enough" and committing to a decision, we free ou